
# Звіт до лабораторної роботи №2
**Тема:** Очищення даних та інженерія ознак  
**Мета:** Підготувати набір даних підприємства до побудови моделей
машинного навчання — очистити, трансформувати та закодувати дані,
сформувати повний набір якісних ознак.

**Варіант 2:** Impact of AI on Students (Замовник: LearnWell University Analytics)



In [51]:
import sqlite3
import pandas as pd
import numpy as np

# Підключення до БД SQLite
conn = sqlite3.connect('student_burnout.db')
df = pd.read_sql_query("SELECT * FROM students_burnout", conn)

print("Розмірність датасету:", df.shape)
df.head(3)

Розмірність датасету: (50000, 17)


,student_id,major_category,year_of_study,pre_semester_gpa,weekly_genai_hours,primary_use_case,prompt_engineering_skill,tool_diversity,paid_subscription,traditional_study_hours,perceived_ai_dependency,institutional_policy,anxiety_level_during_exams,post_semester_gpa,skill_retention_score,burnout_risk_level,timestamp
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,1,8.13,5,Allowed_With_Citation,6,2.393,86.44,High,2025-08-01 00:14:10
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,0,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low,2025-10-15 06:24:48
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,0,10.35,5,Strict_Ban,9,3.499,73.93,Medium,2025-06-07 09:26:34


In [52]:
# 1. Первинна перевірка вихідного набору даних на наявність пропусків
initial_nulls = df.isnull().sum()
print("Кількість пропусків у вихідній таблиці по стовпцях:")
print(initial_nulls)
print("-" * 50)
print(f"Загальна кількість пропусків у вихідних даних: {initial_nulls.sum()}")

Кількість пропусків у вихідній таблиці по стовпцях:
student_id                    0
major_category                0
year_of_study                 0
pre_semester_gpa              0
weekly_genai_hours            0
primary_use_case              0
prompt_engineering_skill      0
tool_diversity                0
paid_subscription             0
traditional_study_hours       0
perceived_ai_dependency       0
institutional_policy          0
anxiety_level_during_exams    0
post_semester_gpa             0
skill_retention_score         0
burnout_risk_level            0
timestamp                     0
dtype: int64
--------------------------------------------------
Загальна кількість пропусків у вихідних даних: 0


In [53]:
# 2. Оскільки у вихідних даних пропусків немає (0) то штучно моделюємо 5% випадкових пропусків (MCAR)
np.random.seed(42)
cols_for_na = ['pre_semester_gpa', 'weekly_genai_hours', 'traditional_study_hours']

for col in cols_for_na:
    mask = np.random.rand(len(df)) < 0.05
    df.loc[mask, col] = np.nan

print("Перевірка після внесення 5% пропусків:")
print(df[cols_for_na].isnull().sum())
print("-" * 50)

# 3. Відновлення  пропущених значень медіаною (медіана стійка до асиметрії розподілу та викидів)
for col in cols_for_na:
    med = df[col].median()
    df[col] = df[col].fillna(med)
    print(f"Стовпець '{col}': заповнено медіаною = {med:.3f}")

print("-" * 50)
print("Перевірка:")
print(df[cols_for_na].isnull().sum())

Перевірка після внесення 5% пропусків:
pre_semester_gpa           2478
weekly_genai_hours         2481
traditional_study_hours    2432
dtype: int64
--------------------------------------------------
Стовпець 'pre_semester_gpa': заповнено медіаною = 3.210
Стовпець 'weekly_genai_hours': заповнено медіаною = 5.780
Стовпець 'traditional_study_hours': заповнено медіаною = 11.170
--------------------------------------------------
Перевірка:
pre_semester_gpa           0
weekly_genai_hours         0
traditional_study_hours    0
dtype: int64


In [54]:
# 1. Сезонність екзаменів з поля timestamp (січень, травень, червень, грудень)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['is_exam_season'] = df['timestamp'].dt.month.isin([1, 5, 6, 12]).astype(int)

# 2. Співвідношення годин використання ШІ до традиційного навчання
df['ai_to_traditional_ratio'] = df['weekly_genai_hours'] / (df['traditional_study_hours'] + 1e-5)

df[['weekly_genai_hours', 'traditional_study_hours', 'ai_to_traditional_ratio', 'is_exam_season']].head(3)

,weekly_genai_hours,traditional_study_hours,ai_to_traditional_ratio,is_exam_season
0,23.31,8.13,2.867155,0
1,1.12,16.65,0.067267,0
2,21.26,10.35,2.054104,1


In [55]:
# Цензурування (Winsorization/Clipping) до меж 1.5 * IQR
# Використовуємо обмеження замість видалення рядків для збереження всіх 50 000 записів для зв'язку 1:1
outlier_cols = ['weekly_genai_hours', 'traditional_study_hours', 'skill_retention_score', 'ai_to_traditional_ratio']

for col in outlier_cols:
    q25 = df[col].quantile(0.25)
    q75 = df[col].quantile(0.75)
    iqr = q75 - q25
    lower_limit = max(0, q25 - 1.5 * iqr)
    upper_limit = q75 + 1.5 * iqr
    df[col + '_clean'] = np.clip(df[col], lower_limit, upper_limit)

print("Аномальні значення успішно цензуровано за межами 1.5*IQR.")

Аномальні значення успішно цензуровано за межами 1.5*IQR.


In [56]:
# 1. Порядкове кодування (Ordinal Encoding)
year_map = {'Freshman': 1, 'Sophomore': 2, 'Junior': 3, 'Senior': 4, 'Graduate': 5}
skill_map = {'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}
target_map = {'Low': 0, 'Medium': 1, 'High': 2}

df['year_of_study_ord'] = df['year_of_study'].map(year_map)
df['prompt_skill_ord'] = df['prompt_engineering_skill'].map(skill_map)
df['burnout_risk_encoded'] = df['burnout_risk_level'].map(target_map)

# 2. One-Hot Encoding для номінальних змінних (drop_first=True для k-1 колонок)
nominal_cols = ['major_category', 'primary_use_case', 'institutional_policy']
df_ohe = pd.get_dummies(df[nominal_cols], drop_first=True, dtype=int)

print("Категоріальні змінні закодовано. Кількість OHE колонок:", df_ohe.shape[1])

Категоріальні змінні закодовано. Кількість OHE колонок: 10


In [57]:
# Стандартизація неперервних ознак (Z-score: z = (x - mean) / std) без використання sklearn
scale_cols = ['pre_semester_gpa', 'weekly_genai_hours_clean', 
              'traditional_study_hours_clean', 'post_semester_gpa', 
              'skill_retention_score_clean', 'ai_to_traditional_ratio_clean']

df_scaled = pd.DataFrame()
for col in scale_cols:
    df_scaled[col + '_scaled'] = (df[col] - df[col].mean()) / df[col].std()

# Формування фінальної таблиці зі збереженням student_id як первинного ключа
features_df = pd.concat([
    df[['student_id', 'burnout_risk_encoded', 'year_of_study_ord', 'prompt_skill_ord',
        'tool_diversity', 'paid_subscription', 'perceived_ai_dependency', 
        'anxiety_level_during_exams', 'is_exam_season']],
    df_ohe,
    df_scaled
], axis=1)

# Запис у нову таблицю бази даних
features_df.to_sql('students_burnout_features', conn, if_exists='replace', index=False)

# Створення унікального індексу для забезпечення зв'язку 1:1
cursor = conn.cursor()
cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_features_student_id ON students_burnout_features (student_id);")
conn.commit()
conn.close()

print(f"Готово! Збережено {len(features_df)} рядків у таблицю 'students_burnout_features' зі зв'язком 1:1!")

Готово! Збережено 50000 рядків у таблицю 'students_burnout_features' зі зв'язком 1:1!


## 1. Звіт про очищення даних
* **Аналіз та моделювання пропусків (Missing Values):** 
  Первинний набір даних містив 0 пропусків на 50 000 спостережень. З метою демонстрації, у числові ознаки `pre_semester_gpa`, `weekly_genai_hours` та `traditional_study_hours` було штучно внесено 5% пропусків типу MCAR (~2 450 значень у кожному).
* **Обґрунтування методу заміни:** 
  Пропущені значення відновлено за допомогою **медіанної імпутації** (`fillna(median)`). Медіана була обрана замість середнього арифметичного, оскільки розподіл годин використання ШІ та навчальних годин має правосторонню асиметрію та поодинокі викиди, на які середнє значення реагує суттєвим зміщенням.
* **Опрацювання аномалій (Outliers):** 
  Проаналізовано міжквартильний розмах (IQR). У змінній `weekly_genai_hours` зафіксовано понад 5% екстремальних значень (до 40 год/тиждень). Застосовано метод цензурування (Winsorization/Clipping) до меж $[Q1 - 1.5 \times IQR; Q3 + 1.5 \times IQR]$. Це усунуло спотворення розподілів без видалення рядків, що критично для збереження розмірності бази даних.

## 2. Обґрунтування методів інженерії даних
* **Категоріальне кодування (Categorical Encoding):**
  * Порядкові ознаки (`year_of_study`, `prompt_engineering_skill`, `burnout_risk_level`) закодовано цілими числами (Ordinal Encoding), що зберігає їхню природну градацію.
  * Номінальні ознаки (`major_category`, `primary_use_case`, `institutional_policy`) закодовано за допомогою One-Hot Encoding з параметром `drop_first=True` ($k-1$ бінарних колонок), щоб запобігти ефекту мультиколінеарності (пастці фіктивних змінних).
* **Створення нових ознак (Feature Engineering):**
  * `ai_to_traditional_ratio`: відношення годин роботи з генеративним ШІ до годин традиційного навчання, що відображає ступінь цифрової залежності студента.
  * `is_exam_season`: бінарна ознака, виділена з дати (місяці сесії: січень, травень, червень, грудень), що моделює сезонне зростання академічного навантаження та тривожності.
* **Масштабування (Feature Scaling):** 
  Застосовано стандартизацію (Z-score: $z = (x - \mu)/\sigma$) для всіх неперервних змінних, що центрує середнє значення в 0 та масштабує дисперсію до 1.

## 3. Організація бази даних та зв'язок 1:1
Підготовлений масив даних записано в нову таблицю `students_burnout_features` бази `student_burnout.db`. Таблиця містить стовпець `student_id` з унікальним індексом, що утворює строгий зв'язок 1:1 з вихідною таблицею `students_burnout`.

## 4. Висновок
Під час виконання лабораторної роботи №2 було повністю реалізовано фазу підготовки даних методології CRISP-DM. Дані позбавлено впливу екстремальних викидів, закодовано та стандартизовано без втрати спостережень. Створена аналітична таблиця повністю готова для тренування моделей машинного навчання в лабораторній роботі №3.